# 14. Redes neuronales en Colab (opción B)

Corre **TabM-Fisher** (modelo original), **TabM**, **TabPFN v2.5** y **XGBoost** con las **mismas particiones**
que el estudio de stacking (10 repeticiones × 5 pliegues, semillas 1000-1009), para compararlos al final con
`src/comparacion_final.py`.

- Protocolo pre-registrado: `experiments/protocolo_redes.md` (congelarlo con un tag **antes** de correr).
- Necesita **GPU**: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
- Los resultados se guardan en Google Drive y la corrida es **reanudable**: si Colab se desconecta, vuelvan a
  ejecutar las celdas y sigue desde el último pliegue guardado.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive (para no perder resultados si Colab se desconecta)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/credit_card_redes'
!mkdir -p "$DRIVE"

## 2. Código del proyecto

Si el repositorio es privado, usen un token personal de GitHub de solo lectura:
`https://<token>@github.com/valeriaflorezs/Credit_Card_Project.git` (no lo guarden en el notebook).

In [ ]:
!git clone -q -b claude/festive-clarke-dm98im https://github.com/valeriaflorezs/Credit_Card_Project.git /content/Credit_Card_Project 2>/dev/null || (cd /content/Credit_Card_Project && git pull -q)
%cd /content/Credit_Card_Project
!git log --oneline -1

In [ ]:
# torch ya viene en Colab; se agregan las librerías del proyecto que usa esta corrida.
!pip install -q scikit-learn==1.9.1 imbalanced-learn==0.14.2 xgboost==3.4.1 xlrd==2.0.2 tabm tabpfn

## 3. Datos (UCI) y dataset procesado del proyecto

In [ ]:
!mkdir -p data/raw
![ -f data/raw/default_of_credit_card_clients.xls ] || (wget -q -O /tmp/uci.zip "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip" && unzip -o -q /tmp/uci.zip -d /tmp/uci && cp "/tmp/uci/default of credit card clients.xls" data/raw/default_of_credit_card_clients.xls)
!python -m src.preprocessing > /dev/null && wc -l data/processed/dataset_final.csv

## 4. Acceso a TabPFN v2.5

1. Entren a https://huggingface.co/Prior-Labs/tabpfn_2_5 con su cuenta y acepten la licencia.
2. Creen un token de lectura en https://huggingface.co/settings/tokens y péguenlo cuando lo pida la celda.

In [ ]:
from huggingface_hub import login
login()

## 5. Resultados en Drive

`results/redes_repeated` queda enlazado a la carpeta de Drive, así cada pliegue terminado se guarda allí.

In [ ]:
!rm -rf results/redes_repeated && ln -s "$DRIVE/redes_repeated" results/redes_repeated && mkdir -p "$DRIVE/redes_repeated"
!ls -la results/redes_repeated/

## 6. Prueba rápida (unos minutos): verifica que los cuatro modelos funcionan

In [ ]:
!python -u -m src.run_stacking_repeated --modelos tabm_fisher tabm tabpfn xgboost --solo-base --muestra 3000 --repeticiones 1 --pliegues 3 --internos 3 --salida results/redes_prueba

## 7. Corrida completa (pre-registrada)

Antes de correr: congelen el protocolo (`git tag protocolo-redes-v1` y `git push origin protocolo-redes-v1`).
La corrida va en segundo plano y guarda cada pliegue en Drive. **Si Colab se desconecta**, vuelvan a
ejecutar las secciones 1 a 5 (sin la prueba rápida) y esta celda: sigue desde el último pliegue guardado.

In [ ]:
# Lanza la corrida EN SEGUNDO PLANO (no bloquea el notebook). Si ya está corriendo, no lanza otra.
import subprocess
if subprocess.run(["pgrep", "-f", "run_stacking_repeated"], capture_output=True).returncode == 0:
    print("Ya hay una corrida en marcha; no se lanza otra.")
else:
    !nohup python -u -m src.run_stacking_repeated --modelos tabm_fisher tabm tabpfn xgboost --solo-base --internos 3 --salida results/redes_repeated >> "$DRIVE/redes_log.txt" 2>&1 &
    print("Corrida lanzada. Log:", DRIVE + "/redes_log.txt")

## 7b. Ver el progreso (se actualiza cada minuto)

Muestra pliegues terminados, tiempo restante, F1 provisional por modelo y el log. También guarda
`progreso_redes.html` en tu Drive. Puedes detener esta celda (■) cuando quieras: **la corrida sigue**.

In [ ]:
import subprocess, time
from pathlib import Path
from IPython.display import HTML, clear_output, display
from src.progreso import generar_html

while True:
    corriendo = subprocess.run(["pgrep", "-f", "run_stacking_repeated"], capture_output=True).returncode == 0
    pagina = generar_html(Path("results/redes_repeated"), Path(DRIVE) / "redes_log.txt", corriendo=corriendo)
    Path(DRIVE, "progreso_redes.html").write_text(pagina, encoding="utf-8")
    clear_output(wait=True)
    display(HTML(pagina))
    if not corriendo:
        break
    time.sleep(60)

## 8. Comparación final con el stacking

Copien el `metricas.csv` de la corrida de stacking de su compañera a `DRIVE/stacking_repeated/metricas.csv`.

In [ ]:
!mkdir -p "$DRIVE/stacking_repeated" && rm -rf results/stacking_repeated && ln -s "$DRIVE/stacking_repeated" results/stacking_repeated
!python -m src.stacking_analysis --salida results/stacking_repeated
!python -m src.comparacion_final --salida "$DRIVE/comparacion_final.csv"